# 06 · Telemetry and billing (Block F)

A provider that charges for data needs to know how much each consumer used. This deployment tracks usage per contract.

## Concept: the telemetry pipeline

```
data planes (provider AND consumer)        record each served request: contract id + response size
        │
telemetry agent (one per participant)     gets a SAS token from the authority's telemetry service,
        │                                  publishes the records to the event broker (Azure Event Hub)
event broker
        │   in production: an Azure Function stores the events; locally: nothing does
telemetry storage (authority)             POST /telemetry-events, kept in billingdb
        │
CSV manager (authority)                   monthly report per participant, cross-checking both sides of every contract
```

Both sides report so that neither can under-report: the report generator compares the consumer's and the provider's records for each contract.

Because the ingestion step does not exist locally, this notebook plays its role:

1. **F3**: store one telemetry event per side of a contract;
2. **F4**: generate the consumer's monthly report;
3. **F5**: download it;
4. **F7**: the error cases, and a security note.

In [1]:
import dataspace as ds

run_id = ds.recall("run_id", produced_by="00_environment.ipynb")

TELEMETRY_EVENTS = f"{ds.BASE_URL}/authority/telemetry-events"
BILLING_REPORTS = f"{ds.BASE_URL}/authority/billing-reports"

contract_id = f"tutorial-{run_id}"   # any id; it only has to be the same on both events

## Concept: who is asking for a report

The CSV manager identifies the caller by a **JWT** whose `roles` claim contains `Participant.<name>`.
Names come from `billingdb.participant_id`, seeded at deployment time with one row per participant (DID, email, name).
`ds.make_jwt` builds one signed with HS256 (HMAC-SHA256 and a shared secret), the same way the E2E suite does.

## Pick a month that has no report yet

A report is generated once per participant and month; asking again returns 409. To keep the notebook re-runnable we look for the first month
(from January 2030) for which downloading the consumer's report still returns 404.

In [2]:
consumer_token = ds.make_jwt({"roles": ["Participant.consumer"]})
print("JWT claims:", ds.decode_jwt_payload(consumer_token))


def report_exists(month, year):
    response = ds.get(BILLING_REPORTS, params={"month": month, "year": year},
                      headers={"Authorization": f"Bearer {consumer_token}"}, quiet=True)
    return response.status_code == 200


month, year = 1, 2030
while report_exists(month, year):
    month += 1
    if month > 12:
        month, year = 1, year + 1
print(f"using {month:02d}/{year}")

JWT claims: {'roles': ['Participant.consumer']}


using 04/2030


## F3 · One telemetry event from each side of the contract

**What:** `POST /authority/telemetry-events`. This is what the ingestion step would do for every record coming out of the event broker.

| Field | Meaning |
|---|---|
| `contractId` | the contract the request belonged to |
| `participantId` | **who reported it**: here once as the consumer, once as the provider |
| `responseStatusCode`, `msgSize` | outcome and size (bytes) of the served request |
| `timestamp` | when it happened; decides which monthly report includes it |

**Returns:** 201.

In [3]:
for number, participant in enumerate(["consumer", "provider"]):
    event = {
        "id": f"{run_id}-{number}",
        "contractId": contract_id,
        "participantId": ds.did(participant),
        "responseStatusCode": 200,
        "msgSize": 2048,
        "csvId": None,
        "timestamp": f"{year}-{month:02d}-15T10:00:00Z",
    }
    ds.post(TELEMETRY_EVENTS, event, expect={201})

POST   /authority/telemetry-events  ->  201
POST   /authority/telemetry-events  ->  201


## F4 · Generate the consumer's report for that month

**What:** `POST /authority/billing-reports` with participant name, month and year.
The generator aggregates that participant's events per contract and looks up the counterparty from the other side's events.
If only one side reported, the counterparty is `N/A`.

**Returns:** 201. The CSV is stored in the report blob storage (Azurite).

In [4]:
report_request = {"participantName": "consumer", "month": month, "year": year}
ds.post(BILLING_REPORTS, report_request, expect={201})

POST   /authority/billing-reports  ->  201


<Response [201]>

## F5 · Download it

**What:** `GET /authority/billing-reports?month=..&year=..` with `Authorization: Bearer <JWT>`.

**Returns:** `text/csv`, one line per contract: `contract_id, counterparty_name, data_transfer_response_status, total_transfer_size_in_kB, total_number_of_events`.

In [5]:
report = ds.get(BILLING_REPORTS, params={"month": month, "year": year},
                headers={"Authorization": f"Bearer {consumer_token}"}, expect={200})
print(report.headers["content-type"])

header, *rows = report.text.strip().splitlines()
print(header)
for row in rows:
    if row.startswith(contract_id):
        print(row)

GET    /authority/billing-reports  ->  200
text/csv
contract_id,counterparty_name,data_transfer_response_status,total_transfer_size_in_kB,total_number_of_events
tutorial-1d0d4e,provider,200,2.00,1


Reading the line: the counterparty is `provider` (taken from the provider's event), status 200, `2048 bytes = 2.00 kB`, 1 event.

## F7 · Error cases, and a security note

- an unknown participant role gets **403**;
- a month without a report gets **404**;
- a token with a **forged signature is accepted**: the CSV manager reads the role claim but does not verify the signature. Anyone can read anyone's report.

In [6]:
unknown_participant = ds.make_jwt({"roles": ["Participant.doesnotexist"]})
forged = consumer_token.rsplit(".", 1)[0] + ".not-a-real-signature"

cases = [
    ("unknown participant", unknown_participant, month, year),
    ("month without report", consumer_token, 1, 1999),
    ("forged signature", forged, month, year),
]
for label, token, report_month, report_year in cases:
    response = ds.get(BILLING_REPORTS, params={"month": report_month, "year": report_year},
                      headers={"Authorization": f"Bearer {token}"}, quiet=True)
    print(f"   {label:22} -> {response.status_code}")

   unknown participant    -> 403
   month without report   -> 404
   forged signature       -> 200


Next: **07_inspection.ipynb**